In [ ]:
# ==========================================
# CELL 1: INSTALLATIONS & SETUP
# ==========================================
!pip install -q torch transformers spacy google-genai pydantic
!python -m spacy download en_core_web_sm

import torch
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google import genai
from google.colab import userdata
from pydantic import BaseModel, Field

# 1. Initialize spaCy (Deterministic Grammar Engine)
print("Loading spaCy deterministic NLP engine...")
nlp = spacy.load("en_core_web_sm")

# 2. Initialize HuggingFace NLI Cross-Encoder (Mathematical Judge)
# We use DeBERTa-v3-small because it is highly accurate for NLI and runs fast on a T4 GPU.
print("Loading NLI Cross-Encoder model (DeBERTa-v3)...")
# Change this line in Cell 1:
model_name = "cross-encoder/nli-deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(model_name)
nli_model.eval()

# 3. Initialize Gemini (The Untrusted Proposer)
api_key = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=api_key)

print("✅ All systems initialized and ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Loading spaCy deterministic NLP engine...
Loading NLI Cross-Encoder model (DeBERTa-v3)...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ All systems initialized and ready.


In [ ]:
# ==========================================
# CELL 2: THE STRICT LLM PROPOSER
# ==========================================
class FeatureProposalBatch(BaseModel):
    features: list[str] = Field(
        description="A list of proposed behavioral features, including both explicit rules and assumed edge cases."
    )

def propose_features(user_prompt: str) -> list[str]:
    """Generates features as declarative facts without adding fluff words."""
    sys_prompt = (
        "You are a Feature Proposer for a formal verification pipeline.\n"
        "Read the informal prompt and generate 6-8 declarative features.\n\n"
        "CRITICAL RULES:\n"
        "1. EXPLICIT FEATURES (Generate 2-3): Extract the exact commands from the prompt and convert them to simple declarative facts WITHOUT adding new descriptive words. "
        "(e.g., if prompt says 'sort an array', output exactly 'The function sorts an array.' Do not add 'version' or 'input').\n"
        "2. ASSUMED FEATURES (Generate 4-5): Propose edge cases, error types, or data types that are completely unstated.\n"
        "3. GRAMMAR: Start every feature with 'The function...'"
       "4. NO PRONOUNS: Never use words like 'it', 'them', or 'this'. Always explicitly name the noun (e.g., 'The function prints the maximum number', NOT 'The function prints it').\n"
        # ... (keep the rest of the rules from before)
    )
    res = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=[sys_prompt, f"User Prompt: '{user_prompt}'"],
        config={
            "response_mime_type": "application/json",
            "response_schema": FeatureProposalBatch,
            "temperature": 0.2
        }
    )
    return FeatureProposalBatch.model_validate_json(res.text).features

In [ ]:
# ==========================================
# CELL 3: THE SMART SYNTACTIC NEGATOR
# ==========================================
def negate_feature(feature_text: str) -> str:
    """Deterministically negates a sentence, preventing double-negatives."""

    # Bug Fix: If the feature is already negative, make it positive
    if "does not " in feature_text or "do not " in feature_text or "not " in feature_text:
        return feature_text.replace("does not ", "does ").replace("do not ", "do ").replace("not ", "").replace("  ", " ")

    doc = nlp(feature_text)
    negated_tokens = []

    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ == "VERB":
            if token.tag_ == "VBZ":
                negated_tokens.extend(["does", "not", token.lemma_])
            elif token.tag_ == "VBD":
                negated_tokens.extend(["did", "not", token.lemma_])
            else:
                negated_tokens.extend(["do", "not", token.text])
        else:
            negated_tokens.append(token.text)

    return " ".join(negated_tokens).replace(" .", ".").strip()

In [ ]:
# ==========================================
# CELL 4: THE NLI DUAL-CHECK MATRIX
# ==========================================
def check_nli(user_prompt: str, feature: str) -> dict:
    """Calculates Entailment using an Instruction-Meaning wrapper."""

    # NEW WRAPPER: Frames the prompt as an instruction, and the feature as its meaning.
    premise = f"Instruction: '{user_prompt}'"
    hypothesis = f"This instruction means that {feature.lower()}"

    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = nli_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits[0], dim=-1).tolist()

    # DeBERTa cross-encoder maps to: [Contradiction, Entailment, Neutral]
    return {
        "contradiction": probs[0],
        "entailment": probs[1],
        "neutral": probs[2]
    }

In [ ]:
# ==========================================
# CELL 5: SMART DETERMINISTIC TREE DECOMPOSER
# ==========================================
def decompose_tree(feature_text: str) -> list[str]:
    """Splits compound features using robust dependency mapping."""
    doc = nlp(feature_text)

    for token in doc:
        # Look for the coordinating conjunction 'and'
        if token.dep_ == "cc" and token.text.lower() == "and":
            head = token.head

            # SPLIT RULE: If it connects to a Verb OR if it is a structural conjunction
            if head.pos_ == "VERB" or head.dep_ == "conj":
                # Part 1: Everything before 'and'
                part1 = doc[0:token.i].text.strip()

                # Part 2: Re-attach the subject (e.g., "The function")
                subjects = [w.text for w in doc if w.dep_ in ("nsubj", "nsubjpass")]
                subj_text = (subjects[0] + " ") if subjects else "The function "

                part2 = subj_text + doc[token.i + 1:].text.strip()

                return [part1, part2]

    # If no valid split is found, return as-is
    return [feature_text]

**test 1**

In [ ]:
# ==========================================
# CELL 6: THE PIPELINE ORCHESTRATOR
# ==========================================
def run_formal_pipeline(user_prompt: str):
    print(f"\n{'='*70}")
    print(f"📥 INFORMAL USER PROMPT:\n\"{user_prompt}\"")
    print(f"{'='*70}\n")

    # 1. Propose Features
    print("⏳ Querying Generative LLM for candidate features...")
    candidate_features = propose_features(user_prompt)

    # --- MISSING BLOCK RESTORED: Show Raw LLM Output ---
    print(f"\n{'-'*70}")
    print("🧠 RAW LLM PROPOSED FEATURES (Untrusted / Potential Hallucinations)")
    print(f"{'-'*70}")
    for i, f in enumerate(candidate_features, 1):
        print(f"  {i}. {f}")

    verified_core = []
    ambiguous_interactive = []

    print(f"\n{'-'*70}")
    print("⚙️ NLI DUAL-CHECK VALIDATION (Mathematical Proofing)")
    print(f"{'-'*70}")

    # NLI Threshold tuned for small cross-encoder models on Colab
    ENTAILED_THRESHOLD = 0.45

    for f in candidate_features:
        neg_f = negate_feature(f)

        score_f = check_nli(user_prompt, f)
        score_neg = check_nli(user_prompt, neg_f)

        p_f_entailed = score_f["entailment"]
        p_neg_entailed = score_neg["entailment"]

        print(f"\n🔍 Candidate: \"{f}\"")
        print(f"   I ⊨ F: {p_f_entailed:.3f} | I ⊨ ¬F: {p_neg_entailed:.3f}")

        # CASE 1: Entailed
        if p_f_entailed > ENTAILED_THRESHOLD and p_neg_entailed < 0.30:
            print("   ✅ STATUS: CASE 1 (Entailed). Verified.")
            verified_core.append(f)

        # CASE 2: Contradicted
        elif p_f_entailed < 0.30 and p_neg_entailed > ENTAILED_THRESHOLD:
            print(f"   🛡️ STATUS: CASE 2 (Contradiction). Accepting ¬F: \"{neg_f}\"")
            verified_core.append(neg_f)

        # OTHER CASES: Ambiguous
        else:
            print("   ⚠️ STATUS: AMBIGUOUS. Triggering spaCy Tree Decomposer...")
            sub_nodes = decompose_tree(f)

            for sub in sub_nodes:
                if sub == f:
                    print(f"      └─ ❓ ATOMIC NEUTRAL: \"{sub}\" (Routed to Human)")
                    ambiguous_interactive.append(sub)
                else:
                    sub_score = check_nli(user_prompt, sub)
                    if sub_score["entailment"] > ENTAILED_THRESHOLD:
                        print(f"      └─ ✅ SUB-ENTAILED: \"{sub}\" (Verified)")
                        verified_core.append(sub)
                    else:
                        print(f"      └─ ❓ SUB-NEUTRAL: \"{sub}\" (Routed to Human)")
                        ambiguous_interactive.append(sub)

    # --- FINAL OUTPUTS ---
    print(f"\n{'='*70}")
    print("🚀 FINAL PIPELINE RESULTS")
    print(f"{'='*70}")

    print("\n✅ VERIFIED CORE CONSTRAINTS (100% Mathematically Proven):")
    if not verified_core:
        print("  (None verified. Try phrasing the explicit rules closer to the prompt.)")
    else:
        for vf in set(verified_core):
            print(f"  • {vf}")

    print("\n👤 INTERACTIVE QUERIES (Human Resolution Required):")
    for idx, af in enumerate(set(ambiguous_interactive), 1):
        print(f"  [Query {idx}] Clarify specification -> \"{af}\"")
    print("\n")

# ==========================================
# RUN THE DEMONSTRATION
# ==========================================
test_prompt = "Write a function to sort an array and return it. It should raise an error if empty."
run_formal_pipeline(test_prompt)


📥 INFORMAL USER PROMPT:
"Write a function to sort an array and return it. It should raise an error if empty."

⏳ Querying Generative LLM for candidate features...

----------------------------------------------------------------------
🧠 RAW LLM PROPOSED FEATURES (Untrusted / Potential Hallucinations)
----------------------------------------------------------------------
  1. The function sorts an array.
  2. The function returns the sorted array.
  3. The function raises an error if the array is empty.
  4. The function sorts the array in ascending order.
  5. The function raises a ValueError when the array is empty.
  6. The function preserves duplicate elements in the sorted array.
  7. The function returns the same array unmodified if the array contains exactly one element.
  8. The function sorts an array of integers.

----------------------------------------------------------------------
⚙️ NLI DUAL-CHECK VALIDATION (Mathematical Proofing)
----------------------------------------

**test 2**

In [ ]:
# ==========================================
# RUN THE DEMONSTRATION
# ==========================================
test_prompt = "Create a function to find the maximum number in a list and print it to the console. Do not return any value."
run_formal_pipeline(test_prompt)


📥 INFORMAL USER PROMPT:
"Create a function to find the maximum number in a list and print it to the console. Do not return any value."

⏳ Querying Generative LLM for candidate features...

----------------------------------------------------------------------
🧠 RAW LLM PROPOSED FEATURES (Untrusted / Potential Hallucinations)
----------------------------------------------------------------------
  1. The function finds the maximum number in a list.
  2. The function prints the maximum number to the console.
  3. The function does not return any value.
  4. The function prints an error message if the list is empty.
  5. The function supports negative numbers and floating-point numbers in the list.
  6. The function throws an exception if the list is null.
  7. The function handles duplicate maximum numbers in the list.

----------------------------------------------------------------------
⚙️ NLI DUAL-CHECK VALIDATION (Mathematical Proofing)
---------------------------------------------

**test3**

In [ ]:
# ==========================================
# RUN THE DEMONSTRATION
# ==========================================
test_prompt = "The function fetches user data from the database."
run_formal_pipeline(test_prompt)


📥 INFORMAL USER PROMPT:
"The function fetches user data from the database."

⏳ Querying Generative LLM for candidate features...

----------------------------------------------------------------------
🧠 RAW LLM PROPOSED FEATURES (Untrusted / Potential Hallucinations)
----------------------------------------------------------------------
  1. The function fetches user data from the database.
  2. The function fetches user data.
  3. The function returns an error when the user data does not exist in the database.
  4. The function returns the user data as a JSON object.
  5. The function throws a connection timeout exception when the database is unreachable.
  6. The function returns null when the requested user identifier is empty.
  7. The function sanitizes the user identifier before querying the database.

----------------------------------------------------------------------
⚙️ NLI DUAL-CHECK VALIDATION (Mathematical Proofing)
-------------------------------------------------------

**see how tree decomposer working**


In [ ]:
# ==========================================
# CELL 7: TREE DECOMPOSER PRESENTATION DEMO
# ==========================================
def run_tree_demo():
    print(f"\n{'='*70}")
    print("🌳 TREE DECOMPOSER ISOLATION TEST")
    print(f"{'='*70}\n")

    # 1. The Setup
    user_prompt = "Write a function to fetch user data."
    print(f"📥 USER PROMPT: \"{user_prompt}\"")

    # 2. The Fake "Bad" LLM Feature
    # Using 'saves' instead of 'formats' to guarantee perfect NLP parsing
    bad_feature = "The function fetches user data and saves it as a JSON object."
    print(f"🤖 BAD LLM GENERATION: \"{bad_feature}\"\n")

    print(f"{'-'*70}")
    print("⚙️ NLI DUAL-CHECK STARTING...")
    print(f"{'-'*70}")

    # 3. Initial NLI Check
    score_f = check_nli(user_prompt, bad_feature)
    print(f"🔍 Testing Full Sentence: \"{bad_feature}\"")
    print(f"   I ⊨ F (Score): {score_f['entailment']:.3f} -> (Below 0.45 Threshold)")
    print("   ⚠️ STATUS: AMBIGUOUS. Triggering spaCy Tree Decomposer...\n")

    # 4. The Tree Decomposer splits it!
    sub_nodes = decompose_tree(bad_feature)

    # 5. Re-testing the Split Nodes
    for i, sub in enumerate(sub_nodes, 1):
        print(f"      └─ ✂️ SPLIT {i}: \"{sub}\"")
        sub_score = check_nli(user_prompt, sub)

        if sub_score["entailment"] > 0.45:
            print(f"         ✅ SUB-ENTAILED: (Score: {sub_score['entailment']:.3f}) -> Rescued & Verified!")
        else:
            print(f"         ❓ SUB-NEUTRAL:  (Score: {sub_score['entailment']:.3f}) -> Routed to Human")

run_tree_demo()


🌳 TREE DECOMPOSER ISOLATION TEST

📥 USER PROMPT: "Write a function to fetch user data."
🤖 BAD LLM GENERATION: "The function fetches user data and saves it as a JSON object."

----------------------------------------------------------------------
⚙️ NLI DUAL-CHECK STARTING...
----------------------------------------------------------------------
🔍 Testing Full Sentence: "The function fetches user data and saves it as a JSON object."
   I ⊨ F (Score): 0.000 -> (Below 0.45 Threshold)
   ⚠️ STATUS: AMBIGUOUS. Triggering spaCy Tree Decomposer...

      └─ ✂️ SPLIT 1: "The function fetches user data"
         ❓ SUB-NEUTRAL:  (Score: 0.007) -> Routed to Human
      └─ ✂️ SPLIT 2: "function saves it as a JSON object."
         ❓ SUB-NEUTRAL:  (Score: 0.000) -> Routed to Human
